### Install Packages

In [0]:
%pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
%pip install matplotlib opencv-python mlflow


### YOLO dataset loader

In [0]:
import os
import torch
from PIL import Image
from torchvision.transforms import functional as F

class YoloDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms=None, num_classes=2):
        self.root = root
        self.transforms = transforms
        self.img_dir = os.path.join(root, "images")
        self.lbl_dir = os.path.join(root, "labels")
        self.imgs = sorted(os.listdir(self.img_dir))
        self.num_classes = num_classes

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        label_name = os.path.splitext(img_name)[0] + ".txt"

        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, label_name)

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f.readlines():
                    cls, x_c, y_c, bw, bh = map(float, line.strip().split())
                    x_c *= w
                    y_c *= h
                    bw *= w
                    bh *= h
                    x1 = x_c - bw / 2
                    y1 = y_c - bh / 2
                    x2 = x_c + bw / 2
                    y2 = y_c + bh / 2
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)  # add +1 for background=0

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        img = F.to_tensor(img)
        return img, target

    def __len__(self):
        return len(self.imgs)


### Load and split data

In [0]:
from torch.utils.data import random_split, DataLoader

dataset = YoloDataset("/dbfs/mnt/lab/unrestricted/rachel.lennon@defra.gov.uk/videos/LabelledFishYOLOv11/train")
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)


### Load Faster RCNN

In [0]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

model = fasterrcnn_resnet50_fpn(pretrained=True)

# Adjust classification head: background + fish (num_classes = 2)
num_classes = 2  # background + 1 class
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


### Train model 

In [0]:
import torch.optim as optim

optimizer = optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()

    lr_scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}")


### ML FLOW

In [0]:
import mlflow.pytorch

with mlflow.start_run():
    mlflow.pytorch.log_model(model, "fasterrcnn_model")
    mlflow.log_param("epochs", num_epochs)
    mlflow.log_metric("final_loss", total_loss)
